Project 4: Reinforcement Learning and the Racetrack Problem

Group 5

Names: Connor Munson, Nick Kasten, Dayton Wickerd

Date: 05 December 2025

Parameters

In [ ]:
GROUP_ID = 5
# SARSA or ValItr or QLrng
ALGORITHM = "SARSA"
TRACK_NAME = "/Provided/W-track.txt"
# STRT or NRST
CRASH_POS = "STRT"


Imports

In [ ]:
import numpy as np
import pandas as pd
import random
import copy
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path

Race Track Builder

In [ ]:
# set of possible accelerations
ACTIONS = [(ax, ay) for ax in (-1, 0, 1) for ay in (-1, 0, 1)]

class RaceTrackEnv:
    def __init__(self, track_file, crashtype, max_speed=5):
        self.max_speed = max_speed
        self.fail_prob = 0.2
        self.crashtype = crashtype
        self.track_importer(track_file)
        self.reset()

    # builds np array from track file
    def track_importer(self, file):
        with open(file, 'r') as f:
            lines = [line.rstrip("\n") for line in f.readlines() if line.strip()]
        
        dims = lines[0].split(',')
        self.height, self.width = int(dims[0]), int(dims[1])

        grid_lines = lines[1:]
        grid = [list(row) for row in grid_lines]
        self.track = np.array(grid)

        self.start_positions = list(zip(*np.where(self.track == 'S')))
        self.finish_positions = list(zip(*np.where(self.track == 'F')))

    # gets the current stae (pos, vel)
    def get_state(self):
        return (self.row, self.col, self.vx, self.vy)

    # reset car to starting line
    def reset(self):
        self.row, self.col = random.choice(self.start_positions)
        self.vx, self.vy = 0, 0
        return self.get_state()

    # apply action at current state and return new state/reward
    def step(self, action):
        ax, ay = action

        # stochastic failure
        if random.random() <= self.fail_prob:
            ax, ay = 0, 0

        # update velocity
        vx = np.clip(self.vx + ax, -self.max_speed, self.max_speed)
        vy = np.clip(self.vy + ay, -self.max_speed, self.max_speed)

        new_row = self.row - vy
        new_col = self.col + vx

        if self.crosses_wall(self.row, self.col, new_row, new_col):
            return self.handle_crash()

        if self.crosses_finish(self.row, self.col, new_row, new_col):
            self.row, self.col, self.vx, self.vy = new_row, new_col, vx, vy
            return self.get_state(), 0, True

        self.row, self.col, self.vx, self.vy = new_row, new_col, vx, vy
        return self.get_state(), -1, False

    def get_start_positions(self):
        return self.start_positions


    # crash handling
    def handle_crash(self):
        if self.crashtype == "NRST":
            self.vx, self.vy = 0, 0
            return (self.row, self.col, 0, 0), -1, False

        elif self.crashtype == "STRT":
            new_state = self.reset()
            return new_state, -1, False

        else: 
            raise ValueError(f"Invalid crash type: {self.crashtype}")

    # checks if car crashes into a wall
    def crosses_wall(self, r1, c1, r2, c2):
        for (r, c) in self.bresenham(int(r1), int(c1), int(r2), int(c2)):
            # check bounds first
            if r < 0 or r >= self.track.shape[0] or c < 0 or c >= self.track.shape[1]:
                return True
            # check wall
            if self.track[r, c] == '#':
                return True
        return False

    # checks if car crosses finish line
    def crosses_finish(self, r1, c1, r2, c2):
        for (r, c) in self.bresenham(int(r1), int(c1), int(r2), int(c2)):
            if self.track[r, c] == 'F':
                return True
        return False

    # bresenham clipping algorithm
    def bresenham(self, r1, c1, r2, c2):
        # avoid corner clipping 
        points = []

        r1, c1, r2, c2 = int(r1), int(c1), int(r2), int(c2)
        dr = abs(r2 - r1)
        dc = abs(c2 - c1)
        sr = 1 if r2 > r1 else -1
        sc = 1 if c2 > c1 else -1

        r, c = r1, c1
        points.append((r, c))

        if dr == 0 and dc == 0:
            return points

        # compute number of steps as max of delta rows/cols
        steps = max(dr, dc)
        for i in range(1, steps + 1):
            r_next = r1 + int(round(i * (r2 - r1) / steps))
            c_next = c1 + int(round(i * (c2 - c1) / steps))
            points.append((r_next, c_next))

            if (r_next != r) and (c_next != c):
                points.append((r_next, c))
                points.append((r, c_next)) 

            r, c = r_next, c_next

        # remove duplicates
        points = list(dict.fromkeys(points))
        return points


Output Builder

In [ ]:
def plot_optimal_path(env, trajectory, save_path):
    track = env.track
    h, w = track.shape

    fig, ax = plt.subplots(figsize=(10, 10))

    # plot walls, start, finish
    for r in range(h):
        for c in range(w):
            cell = track[r, c]
            if cell == "#":
                ax.add_patch(plt.Rectangle((c, h - r - 1), 1, 1, color="black"))
            elif cell == "F":
                ax.add_patch(plt.Rectangle((c, h - r - 1), 1, 1, color="green"))
            elif cell == "S":
                ax.add_patch(plt.Rectangle((c, h - r - 1), 1, 1, color="blue"))

    # unpack trajectory
    xs = [c + 0.5 for (_, c, _, _) in trajectory]
    ys = [h - r - 0.5 for (r, _, _, _) in trajectory]

    # plot the path itself
    ax.plot(xs, ys, linewidth=2, color="red", marker="o")

    ax.set_xlim(0, w)
    ax.set_ylim(0, h)
    ax.set_aspect("equal")
    ax.set_title("Optimal Policy Trajectory")


    plt.grid(False)
    plt.savefig(save_path, dpi=300)
    plt.close()

Helper Functions

In [ ]:
# converts env state to tuple
def state_to_key(state):
    r, c, vx, vy = state
    return (int(r), int(c), int(vx), int(vy))


def clone_env(env, state):
    env_copy = copy.deepcopy(env)
    r, c, vx, vy = state
    env_copy.row = int(r)
    env_copy.col = int(c)
    env_copy.vx = int(vx)
    env_copy.vy = int(vy)
    return env_copy

# builds model for value itration
def build_transition_model(env):
    T = {}
    h, w = env.track.shape

    # All possible states and velocities
    states = []
    for r in range(h):
        for c in range(w):
            if env.track[r, c] == "#":
                continue
            for vx in range(-env.max_speed, env.max_speed + 1):
                for vy in range(-env.max_speed, env.max_speed + 1):
                    states.append((r, c, vx, vy))

    fail_prob = env.fail_prob
    p_succ = 1 - fail_prob
    p_fail = fail_prob

    start_positions = env.start_positions
    n_starts = len(start_positions)

    for s in states:
        for a in ACTIONS:
            entry = []
            next_s, reward, done = simulate_transition(env, s, a)

            if env.crashtype == "STRT" and not done:
                if next_s[:2] in start_positions and next_s[2] == 0 and next_s[3] == 0:
                    for (sr, sc) in start_positions:
                        entry.append((p_succ / n_starts, (sr, sc, 0, 0), reward, False))
                else:
                    entry.append((p_succ, next_s, reward, done))
            else:
                entry.append((p_succ, next_s, reward, done))

            # Failed acceleration
            next_s_fail, reward_fail, done_fail = simulate_transition(env, s, (0, 0))
            entry.append((p_fail, next_s_fail, reward_fail, done_fail))

            T[(s, a)] = entry

    return T, states


def simulate_transition(env, s, a):
    row, col, vx, vy = s
    ax, ay = a

    new_vx = np.clip(vx + ax, -env.max_speed, env.max_speed)
    new_vy = np.clip(vy + ay, -env.max_speed, env.max_speed)

    new_row = row - new_vy
    new_col = col + new_vx

    if env.crosses_wall(row, col, new_row, new_col):
        if env.crashtype == "NRST":
            return (row, col, 0, 0), -1, False
        elif env.crashtype == "STRT":
            new_state = env.reset() 
            return new_state, -1, False

    # Check if finished
    if env.crosses_finish(row, col, new_row, new_col):
        return (new_row, new_col, new_vx, new_vy), 0, True

    return (new_row, new_col, new_vx, new_vy), -1, False


# simulate policy to get trajectory for graphing
def rollout_policy(env, policy, max_steps=1000000):
    
    start_pos = random.choice(env.get_start_positions())
    state = (start_pos[0], start_pos[1], 0, 0)

    trajectory = [state]

    for _ in range(max_steps):
        state_key = state_to_key(state)
        action = policy.get(state_key, (0, 0))

        next_state, reward, done = env.step(action)
        trajectory.append(next_state)

        state = next_state
        if done:
            break

    return trajectory


Value Iteration

In [ ]:
# Value Iteration
def value_iteration(env, gamma=0.97, theta=1e-8, max_iters=500000):
    # build model and get all possible states
    T, states = build_transition_model(env)

    # init value function to zero for all states
    V = {s: 0.0 for s in states}
    # tracks convergence over iterations
    deltas = []

    # run value iteration until convergence or max iterations is reached
    for it in range(max_iters):
        delta = 0.0
        # update value for each state
        for s in states:
            old_v = V[s]

            # find best action by computing Q val for each action
            best_val = -float("inf")
            for a in ACTIONS:
                # expected val for action
                q = sum(p * (r if done else (r + gamma * V[s2])) for (p, s2, r, done) in T[(s, a)])
                best_val = max(best_val, q)

            # update state val with best action val 
            V[s] = best_val
            # tracks max change in value function 
            delta = max(delta, abs(old_v - best_val))

        deltas.append(delta)

        # checks if converged
        if delta < theta:
            print(f"VI converged in {it} iterations.")
            break

    # Build greedy policy
    policy = {}
    for s in states:
        best_a = max(
            ACTIONS,
            key=lambda a: sum(p * (r if done else (r + gamma * V[s2])) for (p, s2, r, done) in T[(s, a)])
        )
        policy[state_to_key(s)] = best_a

    return policy, T, deltas


Q-Learning

In [ ]:
def q_learning(env, episodes=8000, alpha=0.1, gamma=0.95, epsilon=0.1, max_steps=10000):
    # init q table as empty dictionary
    Q = {} 
    # track reward for each episode
    rewards_per_episode = [] 

    # gets Q-val for state-action pair
    def get_Q(s, a):
        return Q.get((s, a), 0.0)

    # run training for num of episodes
    for ep in range(episodes):
        # starts at a random position on starting line with velocity zero
        start_pos = random.choice(env.get_start_positions())
        state = (start_pos[0], start_pos[1], 0, 0)
        s_key = state_to_key(state)

        # tracking reward for this episode
        total_reward = 0  

        # run episode until terminal state for max steps is reached
        for _ in range(max_steps):
            if random.random() < epsilon:
                action = random.choice(ACTIONS)
            else:
                action = max(ACTIONS, key=lambda a: get_Q(s_key, a))

            # take action and see result 
            next_state, reward, done = env.step(action)
            total_reward += reward
            ns_key = state_to_key(next_state)

            # calculates td target using max Q-val of the next state
            max_next_Q = max((get_Q(ns_key, a) for a in ACTIONS), default=0.0)
            td_target = reward + gamma * max_next_Q * (not done)
            td_error = td_target - get_Q(s_key, action)

            # update Q-val 
            Q[(s_key, action)] = get_Q(s_key, action) + alpha * td_error

            s_key = ns_key
            state = next_state

            if done:
                break

        rewards_per_episode.append(total_reward)
        env.reset() 

    # build greedy policy
    policy = {}
    for (s, a), val in Q.items():
        best_val = policy.get(s, (None, -np.inf))[1]
        if val > best_val:
            policy[s] = (a, val)

    # extract only actions
    final_policy = {s: a for s, (a, _) in policy.items()}
    return final_policy, rewards_per_episode


SARSA

In [ ]:
def sarsa(env, episodes=8000, alpha=0.2, gamma=0.97, epsilon=0.1, max_steps=10000):
    # init q table as empty dictionary
    Q = {}
    # track reward for each episode
    rewards_per_episode = []

    # gets Q-val for state-action pair
    def get_Q(s, a):
        return Q.get((s, a), 0.0)

    # run training for num of episodes
    for ep in range(episodes):
         # starts at a random position on starting line with velocity zero
        start_pos = random.choice(env.get_start_positions())
        state = (start_pos[0], start_pos[1], 0, 0)
        s_key = state_to_key(state)

        # choose init action using epsilon-greedy policy 
        if random.random() < epsilon:
            action = random.choice(ACTIONS)
        else:
            action = max(ACTIONS, key=lambda a: get_Q(s_key, a))

        total_reward = 0

        # run episode until terminal state for max steps is reached
        for _ in range(max_steps):
            next_state, reward, done = env.step(action)
            total_reward += reward
            ns_key = state_to_key(next_state)

            # choose next action
            if random.random() < epsilon:
                next_action = random.choice(ACTIONS)
            else:
                next_action = max(ACTIONS, key=lambda a: get_Q(ns_key, a))

            # caculate td target using the next action 
            td_target = reward + gamma * get_Q(ns_key, next_action) * (not done)
            td_error = td_target - get_Q(s_key, action)

            # update Q-val
            Q[(s_key, action)] = get_Q(s_key, action) + alpha * td_error

            s_key = ns_key
            action = next_action
            state = next_state

            if done:
                break

        rewards_per_episode.append(total_reward)
        env.reset()

    # build greedy policy
    policy = {}
    for (s, a), val in Q.items():
        best_val = policy.get(s, (None, -np.inf))[1]
        if val > best_val:
            policy[s] = (a, val)

    final_policy = {s: a for s, (a, _) in policy.items()}
    return final_policy, rewards_per_episode


Grid Searches

In [ ]:
# evaluate policy performance 
def evaluate_policy(env, policy, episodes=500, max_steps=5000):
    successes = 0
    total_steps = 0

    # run policy for num of episodes specified
    for _ in range(episodes):
        eval_env = RaceTrackEnv(env)

        # generate trajectory using current policy 
        trajectory = rollout_policy(eval_env, policy, max_steps=max_steps)

        total_steps += len(trajectory) - 1

        # check if episode ended at finish line
        if eval_env.track[trajectory[-1][0], trajectory[-1][1]] == 'F':
            successes += 1

    # performance metrics 
    avg_steps = total_steps / episodes
    success_rate = successes / episodes
    return avg_steps, success_rate

# grid search for value iteration hyperparameters
def grid_search_value_iteration(env, gammas=[0.95, 0.97, 0.99], thetas=[1e-4, 1e-6, 1e-8]):
    results = []
    # tries all combinations of gamma and theta
    for gamma in gammas:
        for theta in thetas:
            print(f"Running VI with gamma={gamma}, theta={theta}")
            # train policy with current hyperparameters
            policy, _, _ = value_iteration(env, gamma=gamma, theta=theta)
            # eval trained policy and store results
            avg_steps, success_rate = evaluate_policy(env, policy)
            results.append({
                'algorithm': 'ValueIteration',
                'gamma': gamma,
                'theta': theta,
                'avg_steps': avg_steps,
                'success_rate': success_rate
            })
            print(f"Success: {success_rate:.2f}, Avg steps: {avg_steps:.1f}")
    return results

# grid search for value iteration hyperparameters
def grid_search_q_learning(env,
                           alphas=[0.1, 0.2],
                           gammas=[0.95, 0.97],
                           epsilons=[0.05, 0.1],
                           episodes_list=[5000, 8000]):
    results = []
    # tries all combinations of hyperparameters
    for alpha in alphas:
        for gamma in gammas:
            for epsilon in epsilons:
                for episodes in episodes_list:
                    print(f"Q-Learning alpha={alpha}, gamma={gamma}, epsilon={epsilon}, episodes={episodes}")
                    # train policy with current hyperparameters
                    policy, _ = q_learning(env, alpha=alpha, gamma=gamma, epsilon=epsilon, episodes=episodes)
                    # eval trained policy and store results
                    avg_steps, success_rate = evaluate_policy(env, policy)
                    results.append({
                        'algorithm': 'Q-Learning',
                        'alpha': alpha,
                        'gamma': gamma,
                        'epsilon': epsilon,
                        'episodes': episodes,
                        'avg_steps': avg_steps,
                        'success_rate': success_rate
                    })
                    print(f"Success: {success_rate:.2f}, Avg steps: {avg_steps:.1f}")
    return results

# grid search for value iteration hyperparameters
def grid_search_sarsa(env,
                      alphas=[0.1, 0.2],
                      gammas=[0.95, 0.97],
                      epsilons=[0.05, 0.1],
                      episodes_list=[5000, 8000]):
    results = []
    # tries all combinations of hyperparameters
    for alpha in alphas:
        for gamma in gammas:
            for epsilon in epsilons:
                for episodes in episodes_list:
                    print(f"SARSA alpha={alpha}, gamma={gamma}, epsilon={epsilon}, episodes={episodes}")
                    # train policy with current hyperparameters
                    policy, _ = sarsa(env, alpha=alpha, gamma=gamma, epsilon=epsilon, episodes=episodes)
                    # eval trained policy and store results
                    avg_steps, success_rate = evaluate_policy(env, policy)
                    results.append({
                        'algorithm': 'SARSA',
                        'alpha': alpha,
                        'gamma': gamma,
                        'epsilon': epsilon,
                        'episodes': episodes,
                        'avg_steps': avg_steps,
                        'success_rate': success_rate
                    })
                    print(f"Success: {success_rate:.2f}, Avg steps: {avg_steps:.1f}")
    return results


Wrapper

In [ ]:
def main():
    track = RaceTrackEnv(TRACK_NAME, CRASH_POS)
    path = Path(TRACK_NAME)
    end = path.name
    save_path=f"{GROUP_ID}_{ALGORITHM}_{end}_{CRASH_POS}.png"
    if ALGORITHM == "ValItr":
        vpolicy, T, deltas = value_iteration(track)
        vtrajectory = rollout_policy(track, vpolicy)
        plot_optimal_path(track, vtrajectory, save_path)
    elif ALGORITHM == "QLrng":
        qpolicy, rewards = q_learning(track)
        qtrajectory = rollout_policy(track, qpolicy)
        plot_optimal_path(track, qtrajectory, save_path)
    elif ALGORITHM == "SARSA":
        spolicy, rewards = sarsa(track)
        strajectory = rollout_policy(track, spolicy)
        plot_optimal_path(track, strajectory, save_path)

    elif ALGORITHM == "LEARN":
        # Value Iteration grid search
        vi_results = grid_search_value_iteration(track)
        #print(pd.DataFrame(vi_results))

        # Q-Learning grid search
        q_results = grid_search_q_learning(track)

        # SARSA grid search
        sarsa_results = grid_search_sarsa(track)
        #print(pd.DataFrame(sarsa_results)

        all_results = vi_results + q_results + sarsa_results
        df = pd.DataFrame(all_results)
        top5 = df.sort_values(by=['success_rate', 'avg_steps'], ascending=[False, True]) \
         .groupby('algorithm') \
         .head(5)
        # printing top 5 performing parameter sets data for each algorithm
        for algo, group in top5.groupby('algorithm'):
            print("\n=== Top 5 for", algo, "===\n")
        
            cols = [c for c in group.columns if group[c].notnull().any()]
            print(group[cols])
    else:
        print(f"Invalid algorithm selection {ALGORITHM}")

if __name__ == "__main__":
    main()